# Handling Semi Structured Data

In [0]:
df = spark.read.table("retaildataplatform.bronze.cosmosdb_sales_orders")

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
{"_id": {"$oid": "6a999f4a9e0b277853440b4b"}, "clicked_items": [["AVwjhoD2v8e3D1O-nnNv", "35"], ["AVpgQP5vLJeJML43LQbd", "70"], ["AVwjhoD2v8e3D1O-nnNv", "43"], ["AVwjhoD2v8e3D1O-nnNv", "48"], ["AVqVGaCCU2_QcyX9Ozcf", "23"]], "customer_id": 16738781, "customer_name": "GALLARDO,  JOHN D", "number_of_line_items": 2, "order_datetime": 1564684764.0, "order_number": 317568043, "ordered_products": [{"curr": "USD", "id": "AVwjhoD2v8e3D1O-nnNv", "name": "Ankyo - TX 7.2-Ch. Network-Ready A/V Home Theater Receiver - Black", "price": "2036", "promotion_info": null, "qty": "3", "unit": "pcs"}, {"curr": "USD", "id": "AVqVGaCCU2_QcyX9Ozcf", "name": "15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)", "price": "376", "promotion_info": null, "qty": "8", "unit": "pcs"}], "promo_info": []}

In [0]:
customer_id | customer_name         |clicked_items       | score
16738781    | GALLARDO,  JOHN D     |AVwjhoD2v8e3D1O     | 35
16738781    | GALLARDO,  JOHN D     |AVpgQP5vLJeJML43LQbd| 70


-> Array of Array


```
[
    ["AVwjhoD2v8e3D1O-nnNv", "35"], 
    ["AVpgQP5vLJeJML43LQbd", "70"], 
    ["AVwjhoD2v8e3D1O-nnNv", "43"], 
    ["AVwjhoD2v8e3D1O-nnNv", "48"], 
    ["AVqVGaCCU2_QcyX9Ozcf", "23"]
]

```

-> Array of Struct

```
[
    {"curr": "USD", "id": "AVwjhoD2v8e3D1O-nnNv", "name": "Ankyo - TX 7.2-Ch. Network-Ready A/V Home Theater Receiver - Black", "price": "2036", "promotion_info": null, "qty": "3", "unit": "pcs"}, 

    {"curr": "USD", "id": "AVqVGaCCU2_QcyX9Ozcf", "name": "15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)", "price": "376", "promotion_info": null, "qty": "8", "unit": "pcs"}

]



```


-> Array of json

```
"promo_info": [
    {"promo_disc": 0.03, "promo_id": "0", "promo_item": "AVpfMVD-ilAPnD_xW6bu", "promo_qty": "2"}
    ]

```


> 3 Funtions (Most Important)

- `from_json` -> data type (string) -> structured column
- `explode` -> multiple rows
- `explode_outer` -> same as explode and also handle nulls in data

In [0]:
df.printSchema()

In [0]:
'''
{
    "_id": 
    {"$oid": "6a999f4a9e0b277853440b2d"}, 
    
    
    "clicked_items": [["AVpfPEx61cnluZ0-gyT9", "34"], ["AVpfuJ4pilAPnD_xhDyM", "98"], ["AVpe6jFBilAPnD_xQxO2", "60"], ["AVpfIODe1cnluZ0-eg35", "49"]],
    
    
     "customer_id": 19476252, 
     "customer_name": "otbda , outside the box digital agency ,", 
     
     "number_of_line_items": 3,
      "order_datetime": 1564627663.0,
       "order_number": 317568014, 
     
     
     "ordered_products": 
     
     [
         {"curr": "USD", "id": "AVpfuJ4pilAPnD_xhDyM", "name": "Rony LBT-GPX555 Mini-System with Bluetooth and NFC", "price": "993","promotion_info": null,"qty": "3", "unit": "pcs"}, 
       
       {"curr": "USD", "id": "AVpe6jFBilAPnD_xQxO2", "name": "Aeon 71.5 x 130.9 16:9 Fixed Frame Projection Screen with CineWhite Projection Surface", "price": "218", "promotion_info": null, "qty": "3", "unit": "pcs"}, 
       
       {"curr": "USD", "id": "AVpfIODe1cnluZ0-eg35", "name": "Cyber-shot DSC-WX220 Digital Camera (Black)", "price": "448", 
     
     
     "promotion_info": null, "qty": "2", "unit": "pcs"}], 
     
     
     "promo_info": []}
'''




In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    IntegerType,
    ArrayType
)

json_schema =  StructType([
    StructField("_id", StructType([
        StructField("$oid", StringType(), True)
    ]), True),

    StructField(
        "clicked_items",
        ArrayType(
            ArrayType(StringType())
        ),
        True
    ),

    StructField("customer_id", LongType(), True),
    StructField("customer_name", StringType(), True),
    StructField("number_of_line_items", IntegerType(), True),
    StructField("order_datetime", DoubleType(), True),
    StructField("order_number", LongType(), True),

    StructField(
        "ordered_products",
        ArrayType(
            StructType([
                StructField("curr", StringType(), True),
                StructField("id", StringType(), True),
                StructField("name", StringType(), True),
                StructField("price", StringType(), True),
                StructField("promotion_info", StringType(), True),
                StructField("qty", StringType(), True),
                StructField("unit", StringType(), True)
            ])
        ),
        True
    ),

    StructField(
        "promo_info",
        ArrayType(StringType()),
        True
    )
])

In [0]:
import pyspark.sql.functions as F

df_parsed = df.withColumn("data", F.from_json(df["json_data"], json_schema))

In [0]:
df.printSchema()

In [0]:
df_parsed.printSchema()

In [0]:
df_customer = df_parsed.select(
    F.col("data.customer_id").alias("customer_id"),
    F.col("data.customer_name").alias("customer_name"),
    F.col("data.number_of_line_items").alias("number_of_line_items"),
    F.col("data.order_datetime").alias("order_datetime"),
    F.col("data.order_number").alias("order_number"),
    F.explode(F.col("data.ordered_products")).alias("products")
)

In [0]:
df_customer.display()

In [0]:
df_customer.printSchema()

In [0]:
df_customer = df_customer.select(
    F.col("customer_id"),
    F.col("customer_name"),
    F.col("number_of_line_items"),
    F.col("order_datetime"),
    F.col("order_number"),
    F.col("products.curr").alias("curr"),
    F.col("products.id").alias("id"),
    F.col("products.name").alias("name"),
    F.col("products.price").alias("price"),
    F.col("products.promotion_info").alias("promotion_info"),
    F.col("products.qty").alias("qty"),
    F.col("products.unit").alias("unit")
)


df_customer.display()

In [0]:
df_clicked_item = df_parsed.select(
    F.col("data.customer_id").alias("customer_id"),
    F.explode(F.col("data.clicked_items")).alias("clicked_ited")
)
df_clicked_item = df_clicked_item.select(
    F.col("customer_id"),
    F.col("clicked_ited")[0].alias("product_id"),
    F.col("clicked_ited")[1].alias("score")
)

df_clicked_item.display()
